# Unsupervised HP search + multi-seed finals

Hyperparameters are selected by **held-out unsupervised metrics** (Bernoulli LL for LV / LV+$e$ / GNN; negated row MSE for PCA). Ground-truth Hungarian / ARI / NMI are reported only on multi-seed final runs.

Expected artifacts from `run_experiments.py` / `launch_lightning_sweep.py`:
- `hp_results.csv`, `hp_best.json`
- `final_results.csv`, `final_summary.csv`

This notebook adapts to whichever methods are present (e.g. `pca`, `lv`, `lv_e`, `gnn`).

In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (8, 4.5)
plt.rcParams["figure.dpi"] = 120
PALETTE = {
    "pca": "#6c757d",
    "lv": "#e76f51",
    "lv_e": "#2a9d8f",
    "gnn": "#2a6f97",
}
METHOD_ORDER = ["pca", "lv", "lv_e", "gnn"]


def methods_present(df: pd.DataFrame) -> list[str]:
    found = set(df["method"].unique())
    return [m for m in METHOD_ORDER if m in found] + sorted(found - set(METHOD_ORDER))

## 1. Hyperparameter search (unsupervised selection)

In [ ]:
hp = pd.read_csv("hp_results.csv")
best = json.loads(Path("hp_best.json").read_text())
display(hp.sort_values(["method", "val_metric"], ascending=[True, False]).groupby("method").head(5))
print("Selected by val_metric:")
display(pd.DataFrame([
    {"method": m, "name": v["name"], "val_metric": v["val_metric"], **v["hyperparams"]}
    for m, v in best.items()
]))

In [ ]:
methods = methods_present(hp)
n = max(len(methods), 1)
fig, axes = plt.subplots(1, n, figsize=(4.0 * n, 3.5), sharey=False)
if n == 1:
    axes = [axes]
for ax, method in zip(axes, methods):
    sub = hp[hp["method"] == method].copy()
    sns.scatterplot(
        data=sub, x="val_metric", y="gt_hungarian",
        hue="d", style="lr", ax=ax, palette="viridis",
    )
    ax.set_title(f"{method}: val vs GT (GT not used for selection)")
    ax.set_xlabel("held-out val_metric")
    ax.set_ylabel("Hungarian (GT)")
plt.tight_layout()
plt.show()

# LV+e: residual dim / weight decay vs held-out LL (if present)
if "lv_e" in methods and {"d_e", "e_wd"}.issubset(hp.columns):
    sub = hp[hp["method"] == "lv_e"].copy()
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
    sns.scatterplot(data=sub, x="d_e", y="val_metric", hue="e_wd", style="lr", ax=axes[0])
    axes[0].set_title("lv_e: d_e / e_wd vs held-out LL")
    axes[0].set_ylabel("val_metric")
    sns.scatterplot(data=sub, x="val_metric", y="gt_hungarian", hue="d_e", style="e_wd", ax=axes[1])
    axes[1].set_title("lv_e: val vs GT")
    plt.tight_layout()
    plt.show()

## 2. Multi-seed uncertainty (final runs)

In [ ]:
final = pd.read_csv("final_results.csv")
summary = pd.read_csv("final_summary.csv")
display(summary)
display(final.sort_values(["method", "seed"]))

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
order = methods_present(final)
palette = {m: PALETTE.get(m, "#333333") for m in order}
sns.boxplot(
    data=final, x="method", y="gt_hungarian", order=order,
    hue="method", palette=palette, legend=False, ax=ax,
)
sns.stripplot(
    data=final, x="method", y="gt_hungarian", order=order,
    color="black", alpha=0.6, ax=ax,
)
ax.set_ylabel("Hungarian vs visual types")
ax.set_title("Uncertainty over final seeds (best HP per method)")
plt.tight_layout()
plt.show()

for _, row in summary.sort_values("hungarian_mean", ascending=False).iterrows():
    print(
        f"{row['method']}: Hungarian {row['hungarian_mean']:.1f} ± {row['hungarian_std']:.1f} "
        f"(ARI {row['ari_mean']:.3f} ± {row['ari_std']:.3f}, "
        f"NMI {row['nmi_mean']:.3f} ± {row['nmi_std']:.3f})"
    )